In [1]:
from bs4 import BeautifulSoup as BS
import requests
import pandas as pd
from urllib.parse import urljoin


# url of the website to scrape
url ='https://books.toscrape.com/'


# send a GET request to the website
response = requests.get(url, timeout=15)
# set the response encoding to UTF-8
response.encoding = 'utf-8'
# parse the HTML content of the page with BeautifulSoup
soup = BS(response.text, 'html.parser')

In [2]:
# find the first product on the page and extract its name
product = soup.find('article', class_='product_pod')
# extract the product name from the HTML element
product_name = product.find('h3').find('a')['title']
# extract the product price from the HTML element
product_price = product.find('p', class_='price_color').get_text(strip=True)
# convert the product price to a float after removing the currency symbol
product_price = float(product_price.replace('£', ''))
# extract the product availability from the HTML element
product_availability = product.find('p', class_ = 'availability').get_text(strip=True)
# extract the product rating from the HTML element
product_rating = product.find('p', class_='star-rating')
# get the textual representation of the product rating (e.g., 'One', 'Two', etc.)
rating_text = product_rating.get('class')[1]
# map the textual rating to a numerical value
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
# get the numerical rating from the textual representation
rating_number = rating_map[rating_text]
# find all products on the page and store them in a list
products = soup.find_all('article', class_='product_pod')

In [3]:
product_list = []
# iterate over all products and print their names and prices
for product in products:
    product_name = product.find('h3').find('a').get('title')
    product_price = product.find('p', class_='price_color').get_text(strip=True)
    product_price = float(product_price.replace('£', ''))
    product_availability = product.find('p', class_ = 'availability').get_text(strip=True)
    product_rating = product.find('p', class_='star-rating')
    rating_text = product_rating.get('class')[1]
    rating_number = rating_map[rating_text]
    # adds the name, the price, the availability, and the rating to the empty list
    product_list.append({'name': product_name, 
                        'price': product_price,
                        'availability': product_availability,
                        'rating': rating_number})
# function to extract product information from a list of products

In [4]:
def extract_products(products):
    product_list = []
    for product in products:
        product_name = product.find('h3').find('a').get('title')
        product_price = product.find('p', class_='price_color').get_text(strip=True)
        product_price = float(product_price.replace('£', ''))
        product_availability = product.find('p', class_ = 'availability').get_text(strip=True)
        product_rating = product.find('p', class_='star-rating')
        rating_text = product_rating.get('class')[1]
        rating_number = rating_map[rating_text]
        # adds the name, the price, the availability, and the rating to the empty list
        product_list.append({'name': product_name, 
                            'price': product_price,
                            'availability': product_availability,
                            'rating': rating_number,
        })
    return product_list

In [5]:
# extract product information from the second page
extract_product = extract_products(products)

df_products = pd.DataFrame(extract_product)

next_page = soup.find('li', class_='next')
next_page_url = next_page.find('a').get('href')
full_url = urljoin(url, next_page_url)

response_page2 = requests.get(full_url, timeout=15)

soup_page2 = BS(response_page2.content, 'html.parser')

product_page2 = soup_page2.find_all('article', class_='product_pod')

extracted_page2 = extract_products(product_page2)

df_page2 = pd.DataFrame(extracted_page2)

df_all_products = pd.concat(
    [df_products, df_page2],
    ignore_index=True
)


In [6]:
current_url = "https://books.toscrape.com/"
page_number = 1
all_products = []

while page_number:
    response = requests.get(current_url, timeout=15)
    response.raise_for_status()

    soup = BS(response.content, 'html.parser')

    products =soup.find_all(
        'article', class_= 'product_pod'
    )

    extracted_products = extract_products(products)
    all_products.extend(extracted_products)

    print(f"Page {page_number}: {len(extracted_products)} products")

    next_page = soup.find('li', class_='next')

    if next_page is None:
        break

    next_page_url = next_page.find('a').get('href')
    current_url = urljoin(current_url, next_page_url)

    page_number += 1

df_all_products = pd.DataFrame(all_products)

print(f"Total pages: {page_number}")
print(f"Total products: {len(df_all_products)}")
print(df_all_products.shape)

file_name = 'books_catalog.xlsx'

df_all_products.to_excel(file_name, index=False)

if __name__ == "__main__":      # só roda se você executar ESTE arquivo
    
    print(f'File saved as {file_name}')

Page 1: 20 products
Page 2: 20 products
Page 3: 20 products
Page 4: 20 products
Page 5: 20 products
Page 6: 20 products
Page 7: 20 products
Page 8: 20 products
Page 9: 20 products
Page 10: 20 products
Page 11: 20 products
Page 12: 20 products
Page 13: 20 products
Page 14: 20 products
Page 15: 20 products
Page 16: 20 products
Page 17: 20 products
Page 18: 20 products
Page 19: 20 products
Page 20: 20 products
Page 21: 20 products
Page 22: 20 products
Page 23: 20 products
Page 24: 20 products
Page 25: 20 products
Page 26: 20 products
Page 27: 20 products
Page 28: 20 products
Page 29: 20 products
Page 30: 20 products
Page 31: 20 products
Page 32: 20 products
Page 33: 20 products
Page 34: 20 products
Page 35: 20 products
Page 36: 20 products
Page 37: 20 products
Page 38: 20 products
Page 39: 20 products
Page 40: 20 products
Page 41: 20 products
Page 42: 20 products
Page 43: 20 products
Page 44: 20 products
Page 45: 20 products
Page 46: 20 products
Page 47: 20 products
Page 48: 20 products
P

In [7]:
duplicate_count = df_all_products.duplicated().sum()

print(f'Duplicate records: {duplicate_count}')

Duplicate records: 0


In [9]:
missing_value = df_all_products.isna().sum()
display(missing_value)

name            0
price           0
availability    0
rating          0
dtype: int64

In [10]:
display(df_all_products.head())

,name,price,availability,rating
0,A Light in the Attic,51.77,In stock,3
1,Tipping the Velvet,53.74,In stock,1
2,Soumission,50.10,In stock,1
3,Sharp Objects,47.82,In stock,4
4,Sapiens: A Brief History of Humankind,54.23,In stock,5


In [13]:
text_columns = df_all_products.select_dtypes(include='str')

empty_value = text_columns.apply(
    lambda column: column.str.strip().eq('').sum()

)
display(empty_value.to_frame(name='empty_count'))

,empty_count
name,0
availability,0


In [15]:
price_analysis = df_all_products['price'].describe()

display(price_analysis.to_frame(name='price_analysis'))

,price_analysis
count,1000.00000
mean,35.07035
std,14.44669
min,10.00000
25%,22.10750
50%,35.98000
75%,47.45750
max,59.99000


In [16]:

most_expensive = df_all_products.sort_values(
    by="price",
    ascending=False
)

display(most_expensive.head(10))


,name,price,availability,rating
648,The Perfect Play (Play by Play #1),59.99,In stock,3
617,Last One Home (New Beginnings #1),59.98,In stock,3
860,Civilization and Its Discontents,59.95,In stock,2
560,The Barefoot Contessa Cookbook,59.92,In stock,5
366,The Diary of a Young Girl,59.90,In stock,3
657,The Bone Hunters (Lexy Vaughan & Steven Macaul...,59.71,In stock,3
133,Thomas Jefferson and the Tripoli Pirates: The ...,59.64,In stock,1
387,Boar Island (Anna Pigeon #19),59.48,In stock,3
549,The Man Who Mistook His Wife for a Hat and Oth...,59.45,In stock,4
393,The Improbability of Love,59.45,In stock,1
